# 实验：有任务、有状态、有评分、有回归阻断

> 状态：verified；4 个手工教学任务，每个重复 2 次；被测函数是确定性代码，无 LLM。

本实验不称为真实业务基准，也没有独立专家校准集。它演示结果评分、环境重置和已知故障改正，不用于估计真实 Agent 成功率。完整案例：[从任务到回归](../03-cases/01-from-task-dataset-to-regression.md)。

In [1]:
from pathlib import Path
import sys,json
project=Path.cwd().parent/"05-code/eval-harness-python"
sys.path.insert(0,str(project/"src"))
from eval_harness import run_suite,summarize,compare,wilson,pass_at_k
from eval_harness.cli import baseline,candidate,load_tasks
tasks=load_tasks(project/"fixtures/tasks.jsonl")
old=run_suite(tasks,baseline,trials=2)
new=run_suite(tasks,candidate,trials=2)
print(json.dumps({"baseline":summarize(old),"candidate":summarize(new)},ensure_ascii=False,indent=2))

{
  "baseline": {
    "tasks": 4,
    "trials": 8,
    "successes": 2,
    "trial_success_rate": 0.25,
    "macro_task_success_rate": 0.25,
    "system_errors": 0,
    "per_task": {
      "missing-version": {
        "successes": 0,
        "trials": 2
      },
      "old-version": {
        "successes": 0,
        "trials": 2
      },
      "valid": {
        "successes": 2,
        "trials": 2
      },
      "wrong-tenant": {
        "successes": 0,
        "trials": 2
      }
    },
    "slices": {
      "normal": 1.0,
      "permission": 0.0,
      "version": 0.0
    },
    "uncertainty": "Not inferred: repeated deterministic fixtures are not independent population samples."
  },
  "candidate": {
    "tasks": 4,
    "trials": 8,
    "successes": 8,
    "trial_success_rate": 1.0,
    "macro_task_success_rate": 1.0,
    "system_errors": 0,
    "per_task": {
      "missing-version": {
        "successes": 2,
        "trials": 2
      },
      "old-version": {
        "successes": 2,
 

旧版本对所有请求直接返回记录；新版本先检查租户和版本。检查最终 `lookups == 1` 能发现跨 Trial 污染。这里只在 Python 对象层 deepcopy 隔离，不是文件/网络/进程沙箱。

In [2]:
assert all(t.fixture["lookups"]==0 for t in tasks)
assert all(r.state["lookups"]==1 for r in old+new)
failed=next(r for r in old if r.task_id=="wrong-tenant")
print("checks",failed.checks)
print("events",failed.events)
print("candidate events",next(r for r in new if r.task_id=="wrong-tenant").events)

checks {'output:answer': False, 'output:source_id': False, 'output:abstained': False, 'state:lookups': True}
events [{'sequence': 0, 'type': 'trial_start', 'attributes': {'task_id': 'wrong-tenant', 'trial_index': 0}}, {'sequence': 1, 'type': 'lookup', 'attributes': {'scope_checked': False}}, {'sequence': 2, 'type': 'trial_end', 'attributes': {'status': 'completed', 'success': False}}]
candidate events [{'sequence': 0, 'type': 'trial_start', 'attributes': {'task_id': 'wrong-tenant', 'trial_index': 0}}, {'sequence': 1, 'type': 'filter', 'attributes': {'authorized': False, 'current': True}}, {'sequence': 2, 'type': 'trial_end', 'attributes': {'status': 'completed', 'success': True}}]


In [3]:
gate=compare(old,new)
regression=compare(new,old)
print("修复比较",gate)
print("模拟回退",regression)
assert gate["passed"] and not regression["passed"]
assert set(regression["critical_failures"])=={"wrong-tenant","old-version"}

修复比较 {'passed': True, 'regressed_tasks': [], 'improved_tasks': ['missing-version', 'old-version', 'wrong-tenant'], 'critical_failures': [], 'success_rate_delta': 0.75, 'policy': 'strict paired regression; not a statistical non-inferiority test'}
模拟回退 {'passed': False, 'regressed_tasks': ['missing-version', 'old-version', 'wrong-tenant'], 'improved_tasks': [], 'critical_failures': ['old-version', 'wrong-tenant'], 'success_rate_delta': -0.75, 'policy': 'strict paired regression; not a statistical non-inferiority test'}


不要把 8/8 写成“100% 可靠”。本组重复确定性任务不增加新的独立样本。下面 Wilson 和 pass@k 仅用人工数值练习公式，假设适用时才用于推断。

In [4]:
print("假设10个独立同分布试验8次成功，95% Wilson",wilson(8,10))
print("假设10个候选2个成功，选3个至少1个成功",pass_at_k(10,2,3))
assert abs(pass_at_k(10,2,3)-.5333333333333333)<1e-12

假设10个独立同分布试验8次成功，95% Wilson (0.49016247153664183, 0.9433178485456247)
假设10个候选2个成功，选3个至少1个成功 0.5333333333333333


**读结果**：2/8 → 8/8 是修复明确逻辑缺陷后的教学结果；它不证明模型理解权限。若要比较真实 Agent，固定任务、工具、模型与配置，按任务族划分数据，并保持标签不传给被测系统。代码、输出与局限见 [工程说明](../05-code/eval-harness-python/README.md)。